In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install yfinance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import yfinance as yf

# 1. Download Historical Data for Pepsi (PEP) and Coca-Cola (KO)
tickers = ['PEP', 'KO']
data = yf.download(tickers, start='2018-01-01', end='2026-09-10')['Close']

# Drop any missing dates
data = data.dropna()
data.columns = ['KO', 'PEP']

# 2. Cointegration Test (Engle-Granger)
score, pvalue, critical_values = coint(data['PEP'], data['KO'])
print(f"Cointegration Test p-value: {pvalue:.4f}")

if pvalue < 0.05:
    print("Result: PEP and KO are cointegrated! (Reject H0 at 5% significance)")
else:
    print("Result: NOT cointegrated over this period. Proceed with caution.")

# 3. Dynamic Hedge Ratio & Spread Calculation via OLS
X = sm.add_constant(data['KO'])
model = sm.OLS(data['PEP'], X).fit()
hedge_ratio = model.params['KO']
print(f"Hedge Ratio (PEP per 1 KO): {hedge_ratio:.4f}")

# 4. Rolling Z-Score Calculation (20-day window)
data['Spread'] = data['PEP'] - hedge_ratio * data['KO']
lookback = 20
data['Mean'] = data['Spread'].rolling(window=lookback).mean()
data['Std'] = data['Spread'].rolling(window=lookback).std()
data['ZScore'] = (data['Spread'] - data['Mean']) / data['Std']

# 5. Plot the Strategy Spread and Z-Score
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(data['Spread'], label='PEP - (HedgeRatio * KO)', color='purple')
ax1.plot(data['Mean'], label='20-Day Mean', color='orange', linestyle='--')
ax1.set_title('PEP vs KO Spread Dynamics')
ax1.legend()

ax2.plot(data['ZScore'], label='Z-Score', color='blue')
ax2.axhline(1.5, color='red', linestyle='--', label='Upper Entry (+1.5)')
ax2.axhline(-1.5, color='green', linestyle='--', label='Lower Entry (-1.5)')
ax2.axhline(0, color='black', linestyle=':', label='Mean Zero')
ax2.set_title('Trading Signals (Z-Score)')
ax2.legend()

plt.tight_layout()
plt.show()